## Why Add Provenance to Distributions?

In the original ProbPipe design, distributions were primarily *uncertainty containers*.
They supported operations such as sampling and density evaluation, but they did not
retain structural information about how they were created.

For example, if we:

- start with a parametric prior,
- approximate it with samples,
- then convert it back to a parametric form,

the final distribution object contains no record of these intermediate steps.

To address this limitation, I introduced a lightweight **provenance tracking mechanism**.
Each distribution can now carry structured metadata describing:

- the operation that produced it,
- the parent distribution(s),
- and additional operation-specific details.

This transforms distributions from anonymous outputs into
**traceable probabilistic objects**.

## Shift

Previously:
Distribution → Distribution → Distribution  

Now:
Each distribution forms a node in a provenance DAG.

This does not yet implement full probabilistic program tracing
(e.g., prior + likelihood graphs, pinning, joint distributions).
Instead, this is a minimal first step:

Every distribution remembers the probabilistic operation that created it.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Mapping, Sequence, Optional, Generic, TypeVar, Callable
import uuid
from abc import ABC, abstractmethod
import numpy as np
from probpipe import Gaussian

from probpipe.custom_types import Array, ArrayLike, PRNG, Float, Number
from probpipe.array_backend.utils import _ensure_matrix, _ensure_vector
from probpipe.linalg.linear_operator import LinOp, RootLinOp

NumberT = TypeVar("NumberT", bound=Number)


# TODO: look at existing provenance implementations!!
@dataclass(frozen=True)
class Provenance:
    op: str                           # e.g. "prior", "transform", "approximate" (this can be seperate), "convert" (conversion but if it is an appriximate or not)
    parents: tuple["Provenance", ...] = () #XXX: storing all parents or the most immediate one???
    details: Mapping[str, Any] = field(default_factory=dict) 
    uid: str = field(default_factory=lambda: uuid.uuid4().hex)

    def chain(self) -> Sequence["Provenance"]:
        """Return a linearized view (not necessarily unique) for quick debugging."""
        out = []
        stack = [self]
        seen = set()
        while stack:
            node = stack.pop()
            if node.uid in seen:
                continue
            seen.add(node.uid)
            out.append(node)
            stack.extend(node.parents)
        return out

In [15]:
# Our original Distribution class but with __init__ and source method. 
class Distribution(Generic[NumberT], ABC):
    def __init__(self, *, source: Provenance | None = None):
        self._source = source

    @property
    def source(self) -> Provenance | None:
        return self._source
    
    def _with_source(self, op: str, *, parents: list["Distribution"] = None, **details):
        p = tuple(d.source for d in (parents or []) if d.source is not None)
        self._source = Provenance(op=op, parents=p, details=details)
        return self

    
    def sample(self, n_samples: int = 1) -> Array[NumberT]:
        raise NotImplementedError("This method may be implemented by subclasses (optional)")
    
    def density(self, x: Array[NumberT]) -> Array[Float]:
        raise NotImplementedError("This method may be implemented by subclasses (optional)")
    

    def expectation(self, func: Callable[[Array[NumberT]], Array]) -> Distribution:
        raise NotImplementedError("This method may be implemented by subclasses (optional)")

    @classmethod
    @abstractmethod
    def from_distribution(cls, other: Distribution, **fit_kwargs: Any) -> Distribution[NumberT]:
        raise NotImplementedError("This method should be implemented by subclasses")
    


In [16]:
## our original EmpiricalDistribution class but with a new from_distribution class method.
class EmpiricalDistribution(Distribution):
    def __init__(
        self,
        x: Array,
        weights: Array | None = None,
        *,
        rng: PRNG | None = None,
    ):
        X = _ensure_matrix(x, as_row_matrix=True)
        n, d = X.shape
        if n < 1:
            raise ValueError("EmpiricalDistribution requires at least one sample.")

        if weights is None:
            w = np.full(n, 1.0 / n, dtype=X.dtype)
        else:
            w = _ensure_vector(weights, as_column=False, length=n)
            if np.any(w < 0):
                raise ValueError("weights must be nonnegative.")
            s = w.sum()
            if s <= 0:
                raise ValueError("weights cannot all be zero.")
            w = w / s

        self._X = X.astype(float)
        self._w = w.astype(float)
        self._n = int(n)
        self._d = int(d)
        self._rng = rng or np.random.default_rng()

        # Compute empirical mean and covariance.
        self._mean = (self._w[:, np.newaxis] * self._X).sum(axis=0)
        cov_root = ((self._X - self._mean) * np.sqrt(self._w)[:, np.newaxis]).T  # (d,n)
        self._cov = RootLinOp(cov_root)

        # cumulative weights for fast inverse-transform resampling
        self._cw = np.cumsum(self._w)

    @property
    def n(self) -> int:
        """Number of stored samples."""
        return self._n

    @property
    def dim(self) -> int:
        """Dimensionality."""
        return self._d

    @property
    def samples(self) -> Array:
        """A view of the stored samples, shape (n, d)."""
        return self._X

    @property
    def weights(self) -> Array:
        """A view of normalized weights, shape (n,)."""
        return self._w

    def mean(self) -> Array:
        """Weighted mean, shape (d,)."""
        return self._mean

    def cov(self) -> LinOp:
        """Weighted *population* covariance, shape (d, d)."""
        return self._cov

    def var(self) -> Array:
        """Weighted population variance per dimension, shape (d,)."""
        return self._cov.diag()

    def std(self) -> Array:
        """Weighted population standard deviation per dimension, shape (d,)."""
        return np.sqrt(np.maximum(self.var(), 0.0))

    def __str__(self) -> str:
        """String representation of the empirical distribution."""
        # Check if weights are uniform
        uniform_weights = np.allclose(self._w, 1.0 / self._n)
        weight_info = "uniform" if uniform_weights else "weighted"

        # Format mean and std for display
        if self._d == 1:
            mean_str = f"{self._mean[0]:.4g}"
            std_str = f"{self.std()[0]:.4g}"
        elif self._d <= 3:
            mean_str = "[" + ", ".join(f"{m:.4g}" for m in self._mean) + "]"
            std_str = "[" + ", ".join(f"{s:.4g}" for s in self.std()) + "]"
        else:
            mean_str = f"[{self._mean[0]:.4g}, ..., {self._mean[-1]:.4g}]"
            std_str = f"[{self.std()[0]:.4g}, ..., {self.std()[-1]:.4g}]"

        return (
            f"EmpiricalDistribution(n={self._n}, dim={self._d}, {weight_info}, "
            f"mean={mean_str}, std={std_str})"
        )

    def sample(self, n_samples: int = 1, *, replace: bool = True) -> Array:
        """
        Resample draws from the empirical distribution with replacement,
        using the stored weights. Returns shape (n_samples, d).
        """
        n_samples = int(n_samples)
        if not replace and n_samples > self._n:
            raise ValueError("Cannot sample more than n without replacement.")
        idx = self._rng.choice(self._n, size=n_samples, replace=replace, p=self._w)
        return self._X[idx]

    rvs = sample

    def density(self, x: Array) -> Array:
        """Approximate density using a Gaussian fit to the empirical samples. See log_density."""
        return np.exp(self.log_density(x))

    def log_density(self, x: Array) -> Array:
        """
        Approximate log density using a Gaussian fit to the empirical samples.
        """
        from ..probpipe.distributions.real_vector.gaussian import Gaussian
        return Gaussian(mean=self._mean, cov=self._cov.to_dense()).log_density(x)

    # TODO: come back to this:
    def expectation(
        self,
        func: Callable[[Array], Array],
        *,
        n_mc: int = 2048,
    ):
        raise NotImplemented

    # THIS IS NEW
    @classmethod
    def from_distribution(cls, other: Distribution, **fit_kwargs: Any) -> "EmpiricalDistribution":
        n = fit_kwargs.get("num_samples", 2048)
        samples = other.sample(n)

        src = Provenance(
            op="approximate",
            parents=tuple([other.source] if other.source is not None else []),
            details={"method": "empirical", "num_samples": n, "cls": cls.__name__},
        )
        return cls(samples, source=src)

## Example 2: Conversion with Provenance Tracking

We demonstrate provenance tracking through the following workflow:

1. Create a parametric Gaussian prior.
2. Approximate it using an empirical (sample-based) distribution.
3. Convert the empirical distribution back into a Gaussian using moment matching.

The key idea is that the final Gaussian distribution
should retain structured information about:

- the conversion step,
- the empirical approximation step,
- and the original prior.


In [ ]:
# prior -> approximate -> convert

# We construct a Gaussian prior and explicitly label it as a "prior" in the provenance metadata.
prior = Gaussian(mean=np.zeros(2), cov=np.eye(2))._with_source(op="prior", name="w")

# We approximate the Gaussian prior using Monte Carlo sampling.
emp = EmpiricalDistribution.from_distribution(prior, num_samples=4096)

# Convert empirical back to parametric 
approx_gauss = Gaussian.from_distribution(emp, method="moment_match")

#The resulting Gaussian distribution records:
print(approx_gauss.source.op) #that it was produced via conversion,
print([p.op for p in approx_gauss.source.parents]) #its parent distribution


AttributeError: 'Gaussian' object has no attribute '_with_source'

## Why this matters Before Integrating TFP? 

By first introducing provenance tracking, we ensure that ProbPipe treats distributions as structured, traceable probabilistic objects.

Once this abstraction layer is stable we can integrate as TFP joint distributions or transformation systems.
